# Measuring the angular two-point correlation function of CMASS

This notebook walks you through measuring how galaxies cluster on the sky, using the BOSS **CMASS** sample and a matching random catalog. You'll count galaxy pairs with **TreeCorr**, combine them with the **Landy–Szalay estimator**, and end up with the angular two-point correlation function $w(\theta)$ — the same measurement you'll later run on your DMASS sample variants.

By the end you should be able to (1) explain in your own words what $w(\theta)$ measures, (2) say why a random catalog is needed, and (3) run the measurement yourself.

## Watch this first

Before any definitions, watch this short animation (about 20 seconds) to get a feel for what we're measuring. It picks pairs of galaxies, measures how far apart each pair is, and tallies the separations into a histogram. The pile-up of *close* pairs — more than you'd get from a random scatter — is the clustering signal. Everything below just makes this precise.

The animation uses a physical "separation." In this project we work on the sky, so our separation is the **angle** $\theta$, and the result is written $w(\theta)$.

> **Video credit:** CAASTRO (ARC Centre of Excellence for All-Sky Astrophysics), via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Two-point-correlation-function-astronomy.webm).

In [ ]:
import os
from IPython.display import Video, HTML, display

VIDEO_FILE = "Two-point-correlation-function-astronomy.webm.720p.vp9.webm"
PAGE_URL   = "https://commons.wikimedia.org/wiki/File:Two-point-correlation-function-astronomy.webm"

if os.path.exists(VIDEO_FILE):
    display(Video(VIDEO_FILE, embed=True, width=720))
    display(HTML(f'<p style="font-size:14px">Not playing in this editor? '
                 f'<a href="{PAGE_URL}" target="_blank">▶ Watch on Wikimedia Commons</a> (CAASTRO).</p>'))
else:
    display(HTML(f'<p style="font-size:14px"><b>File not found:</b> <code>{VIDEO_FILE}</code><br>'
                 f'Looked in: <code>{os.getcwd()}</code><br>'
                 f'Put the .webm in this folder, or '
                 f'<a href="{PAGE_URL}" target="_blank">▶ watch on Wikimedia Commons</a> (CAASTRO).</p>'))

## 1. What is the two-point function?

Now that you've seen the idea, let's make it precise. Galaxies are not scattered randomly across the sky. They trace the cosmic web — clusters, filaments, and voids — so you are *more* likely to find a galaxy near another galaxy than you would be if they were thrown down at random.

The **angular two-point correlation function** $w(\theta)$ puts a number on that "more likely." It is the *excess probability*, compared to a random distribution, of finding a pair of galaxies separated by an angle $\theta$ on the sky:

$$ dP = \bar{n}^2\,\big[\,1 + w(\theta)\,\big]\, d\Omega_1\, d\Omega_2 $$

where $\bar{n}$ is the mean number of galaxies per unit area. Reading the three cases:

- $w(\theta) = 0$ → galaxies at separation $\theta$ are distributed like random (no clustering).
- $w(\theta) > 0$ → more pairs than random at that separation → the sample is clustered.
- $w(\theta) = 1$ → twice as many pairs as random at that separation.

So $w(\theta)$ is just a curve: how much more clustered than random, as a function of angular separation. Further down you'll plot exactly this for CMASS.

## 2. Why we need a random catalog

The histogram in the animation was the raw **data–data pair count (DD)**. By itself it is *not* the correlation function — even a perfectly random catalog produces a sloped histogram, simply because the survey has edges and there are more ways to make pairs at some separations than others. To isolate real clustering we have to divide that geometry out, and that's exactly what a random catalog is for.

A real survey doesn't cover a clean rectangle of sky. It has a footprint with ragged edges and holes (masks) where bright stars, bad CCD regions, or imaging problems force us to drop galaxies. If we only counted data pairs, we couldn't separate real clustering from these geometry effects.

A **random catalog** fixes this. It is a set of points with *no* intrinsic clustering, spread over *exactly* the same footprint and mask as the data. It answers the question: "how many pairs at separation $\theta$ would I get if there were no clustering at all, given this survey's shape?" That is the baseline we compare the data against. When you plot the data and randoms side by side below, look for the masked holes in the random panel — that geometry is what the randoms encode.

## 3. The Landy–Szalay estimator

To turn pair counts into $w(\theta)$ we count three kinds of pairs in each angular bin (each normalized by the total number of possible pairs of that type):

- **DD** — data–data pairs (galaxy with galaxy)
- **RR** — random–random pairs (random point with random point)
- **DR** — data–random pairs (galaxy with random point)

The simplest estimator is $w = DD/RR - 1$, but the field standard is the **Landy–Szalay** estimator:

$$ w(\theta) = \frac{DD - 2\,DR + RR}{RR} $$

The extra $DR$ cross-term makes the estimator far less sensitive to edges and masks, and gives it close to the lowest variance of the common estimators — which is why it's the standard choice. You'll see why in the Landy & Szalay (1993) reading. In TreeCorr, calling `calculateXi(rr=rr, dr=dr)` applies exactly this formula.

## 4. Setup

In [ ]:
import scipy
import numpy as np
import matplotlib.pyplot as plt
import fitsio
import treecorr

# Make plots look nicer
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 13


%matplotlib inline
%load_ext autoreload
%autoreload 2

## 5. Load the galaxy sample and the random catalog

We use the BOSS DR12 **CMASS South** galaxy catalog and its matching random catalog. Each catalog is a table with one row per object; the columns we need are `RA` and `DEC` (sky coordinates, in degrees). The CMASS catalog also carries weight columns, which we'll use below.

The file paths point to the shared project space on OSC.

The random catalog is large. If plotting or measuring is slow while you're experimenting, you can thin the randoms — but use the **full** catalog for your final numbers, since more randoms means less noise in RR.

In [ ]:
cmass_filename = '/fs/ess/PHS0411/summer/galaxy_DR12v5_CMASS_South_Photo.fits.gz'
cmass = fitsio.read(cmass_filename)
random_filename = '/fs/ess/PHS0411/summer/random0_DR12v5_CMASS_South.fits.gz'
random = fitsio.read(random_filename)

## 6. Look at the data first

Before measuring anything, plot the two catalogs. On the left (CMASS) you should see structure — denser regions and emptier voids. On the right (random) the points are featureless except for the masked holes. The random panel looks denser because we deliberately use many more random points than galaxies (about 50 times denser than CMASS), which beats down the noise in the RR counts.

In [ ]:
fig, (ax, ax2) = plt.subplots(1,2, figsize=(10, 5))

ax.scatter(cmass['RA'], cmass['DEC'], s=3, alpha=0.5, color='tomato', rasterized=True)
ax2.scatter(random['RA'], random['DEC'], s=2, alpha=0.1, color='tab:blue', rasterized=True)

ax.set_xlabel('Right Ascension (deg)')
ax.set_ylabel('Declination (deg)')
ax.set_title('CMASS')
ax.set_xlim(0,5)
ax.set_ylim(0,5)

ax2.set_xlabel('Right Ascension (deg)')
ax2.set_ylabel('Declination (deg)')
ax2.set_title('Random')
ax2.set_xlim(0,5)
ax2.set_ylim(0,5)

plt.tight_layout()
plt.show()

## 7. Measuring $w(\theta)$ with TreeCorr

Counting every pair of galaxies by brute force is an $O(N^2)$ job — hopeless for millions of objects. **TreeCorr** organizes the catalog into a tree (a hierarchy of cells on the sky) and counts pairs in groups, bringing this down to roughly $O(N\log N)$. For us it does three things: builds the catalogs, counts DD / DR / RR pairs in angular bins, and combines them with Landy–Szalay.

A few details in the code below:

- `NNCorrelation` is TreeCorr's **count–count** correlation (positions only, no shapes) — exactly what $w(\theta)$ needs.
- We split the sky into patches (`npatch=30`) so TreeCorr can estimate errors by **jackknife** (leave-one-patch-out), which captures sample variance rather than just shot noise.
- **Weights.** Each galaxy sample carries its *own* set of weights, tailored to how that sample was built and observed — there is no universal weight. For CMASS, the weights correct for observational systematics (e.g. seeing and stellar density across the sky) and for galaxies the spectrograph missed. We combine them into the standard CMASS completeness weight
$$ 
w_{\rm tot} = w_{\rm systot}\,(\,w_{\rm cp} + w_{\rm noz} - 1\,) 
$$ 
and apply it to the data; the randoms get the FKP weight. Your **DMASS** sample will have its own, *different* weights — so don't assume CMASS's weight columns carry over to DMASS. You don't need to understand the weights in detail right now; for each sample, just apply the weights that come with it. If you're curious, the CMASS weights are described in Lee et al. (2019) and references therein.

TreeCorr docs: [home](https://rmjarvis.github.io/TreeCorr/) · [Getting Started guide](https://rmjarvis.github.io/TreeCorr/_build/html/guide.html) · [`NNCorrelation` reference](https://rmjarvis.github.io/TreeCorr/_build/html/nn.html)

In [ ]:
weight = cmass['WEIGHT_SYSTOT']*( cmass['WEIGHT_CP'] + cmass['WEIGHT_NOZ'] - 1.0 )
weight_random = random['WEIGHT_FKP']

data = treecorr.Catalog(ra=cmass['RA'], dec=cmass['DEC'], 
                        w = weight, ra_units='deg', dec_units='deg', npatch=30)
rand = treecorr.Catalog(ra=random['RA'], dec=random['DEC'], 
                        is_rand=True, w = weight_random, ra_units='deg', dec_units='deg', patch_centers=data.patch_centers)

## 8. Count pairs and apply Landy–Szalay

`process()` does the pair counting; `calculateXi(rr=RR, dr=DR)` applies the Landy–Szalay formula from Section 3. We use **logarithmic** angular bins from `min_sep` to `max_sep` because clustering spans a wide range of scales. With patches set, `estimate_cov(method='jackknife')` gives the error bars.

In [ ]:

nbins = 20 # number of bins in angular separation
# range of angular separations, in arcmin ( 60 arcmin = 1 degree ). 
sep_units = 'arcmin'
min_sep = 2.5 # minimum angular separation in arcmin (leftmost point on x-axis). 
max_sep = 250 # maximum angular separation in arcmin (rightmost point on x-axis)
var_method='jackknife'
verbose=2

# DD pair counts, DR pair counts, and RR pair counts in the Landy-Szalay estimator
DD = treecorr.NNCorrelation(nbins = nbins, max_sep = max_sep, min_sep= min_sep, sep_units=sep_units, var_method=var_method, verbose=verbose)
DR = treecorr.NNCorrelation(nbins = nbins, max_sep = max_sep, min_sep= min_sep, sep_units=sep_units, verbose=verbose)
RR = treecorr.NNCorrelation(nbins = nbins, max_sep = max_sep, min_sep= min_sep, sep_units=sep_units, verbose=verbose)

DD.process(data)
DR.process(data, rand)
RR.process(rand)

# angular separation theta is stored in DD.meanr
theta = DD.meanr
# angular correlation function w(theta) is calculated using the Landy-Szalay estimator
wtheta, _ = DD.calculateXi(rr=RR,dr=DR)

# error on w(theta)
cov = DD.estimate_cov(method=var_method, cross_patch_weight='match')
err_wtheta = np.sqrt(cov.diagonal())

## 9. Plot and interpret

Plot $w(\theta)$ on a log-$\theta$ axis. You should see the clustering signal rising toward smaller separations — galaxies are more strongly correlated on small scales.

This is the measurement you'll repeat for each DMASS variant. When you compare a variant to CMASS, measure **both the same way** (same weights, same binning) so that any difference reflects the *selection*, not the analysis choices.

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(8,6))

ax.errorbar(theta, wtheta, yerr=err_wtheta, fmt='.', capsize=5,  color='blue')
ax.set_xscale('log')  # Plot on log scale
ax.set_xlabel('$\\theta$ (arcmin)')
ax.set_ylabel('$\\omega(\\theta)$')
ax.set_title('angular correlation function of CMASS galaxies')


## 10. Compute $w(\theta)$ for your divided samples

Earlier, in the **CMASS division test**, you split CMASS into subsamples — for example by $i$-band magnitude, or by color. Now put everything together: measure $w(\theta)$ for each of those subsamples using the exact same machinery as above, and see how the clustering changes from one subsample to the next.

To avoid copy-pasting the whole TreeCorr block for every subsample, wrap it in a small function. It reuses the random catalog and binning you already set up:

In [ ]:
# Example: split CMASS into bright vs faint in i-band magnitude.
# Use the SAME column and selection you built in the CMASS division test --
# the column name 'mag_i' below is a placeholder, so check your catalog's columns.
mask_low_i = (cmass['mag_i'] < 19.3) 
mask_high_i = (cmass['mag_i'] >= 19.5) 

cmass_low_i = cmass[mask_low_i]
cmass_high_i = cmass[mask_high_i]

For the color cuts: 

In [ ]:
mask_low_ri = (cmass['r-i'] < 0.9) 
mask_high_ri = (cmass['r-i'] >= 1.0) 

cmass_low_ri = cmass[mask_low_ri]
cmass_high_ri = cmass[mask_high_ri]

Repeat the same two steps — build the subsample, then compute the two point function for your **color-selected** subsamples and any other divisions you made in the division test, and overplot them.

Then answer (bring these to our meeting):

- Do brighter or fainter galaxies cluster more strongly? Redder or bluer?
- Which subsample has the largest $w(\theta)$, and at which scales?
- Can you explain the differences physically — what do they suggest about the kinds of dark matter halos these galaxies live in?

## Helpful links and references

- **TreeCorr** — [documentation home](https://rmjarvis.github.io/TreeCorr/), [Getting Started guide](https://rmjarvis.github.io/TreeCorr/_build/html/guide.html), [`NNCorrelation` reference](https://rmjarvis.github.io/TreeCorr/_build/html/nn.html)
- **Landy, S. D. & Szalay, A. S. (1993)**, "Bias and Variance of Angular Correlation Functions," *ApJ* 412, 64 — the estimator we use here. [ADS](https://ui.adsabs.harvard.edu/abs/1993ApJ...412...64L)
- **Correlation function (astronomy)** — plain-language overview. [Wikipedia](https://en.wikipedia.org/wiki/Correlation_function_(astronomy))
- **Animation** — two-point correlation function, CAASTRO, via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Two-point-correlation-function-astronomy.webm)
- **Reid, B. et al. (2016)**, "SDSS-III BOSS DR12: galaxy target selection and large-scale structure catalogues," *MNRAS* 455, 1553 — defines the CMASS weights ($w_{\rm systot}$, $w_{\rm cp}$, $w_{\rm noz}$, $w_{\rm FKP}$). [ADS](https://ui.adsabs.harvard.edu/abs/2016MNRAS.455.1553R)
- **Lee, S. et al. (2019)**, "Producing a BOSS CMASS sample with DES imaging," *MNRAS* 489, 2887 — the DMASS sample and its own weights. [ADS](https://ui.adsabs.harvard.edu/abs/2019MNRAS.489.2887L)